<a href="https://colab.research.google.com/github/jmmount/GrahamsNumber/blob/main/grhm_num.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import yfinance as yf
import math


def calculate_graham_number(ticker_symbol):
    """
    Calculates the Graham Number: sqrt(22.5 * EPS * BVPS)
    using Yahoo Finance data.
    """
    try:
        # Initialize the Ticker object
        ticker = yf.Ticker(ticker_symbol)
        info = ticker.info

        # 1. Fetch EPS (Trailing Twelve Months)
        eps = info.get('trailingEps')

        # 2. Fetch Book Value Per Share
        bvps = info.get('bookValue')

        # Validation: Graham Number is not applicable if EPS or BVPS are negative
        if eps is None or bvps is None:
            return f"Data missing for {ticker_symbol} (EPS: {eps}, BVPS: {bvps})"

        if eps <= 0 or bvps <= 0:
            return f"Calculation invalid for {ticker_symbol}: EPS or BVPS is negative."

        # 3. Apply the Formula: sqrt(22.5 * EPS * BVPS)
        graham_number = math.sqrt(22.5 * eps * bvps)

        return round(graham_number, 2)

    except Exception as e:
        return f"An error occurred: {e}"

# Example Usage:
ticker = "amzn"
result = calculate_graham_number(ticker)
print(f"The Graham Number for {ticker} is: {result}")

The Graham Number for amzn is: 87.92


In [ ]:
import pandas as pd
import yfinance as yf

def get_historical_dividends(ticker_symbol):
    """
    Fetches historical dividend payments, dates, and calculates the dividend
    as a percentage of the stock price at the time of payment.
    """
    try:
        ticker = yf.Ticker(ticker_symbol)
        dividends = ticker.dividends

        if dividends.empty:
            return f"No dividend data found for {ticker_symbol}."

        # Convert the Series to a DataFrame
        dividends_df = dividends.reset_index()
        dividends_df.columns = ['Date', 'Dividend']

        # Fetch historical prices for the dividend dates
        # Using 'period=max' to get all available data to cover all dividend dates
        hist_prices = ticker.history(period="max")
        hist_prices = hist_prices.reset_index()
        hist_prices['Date'] = hist_prices['Date'].dt.tz_convert(None) # Remove timezone for merging

        # Merge dividends with historical prices on date
        # We need to ensure the dates match, often the dividend date is ex-dividend date
        # We'll use a forward fill or closest date merge if direct match is not found.
        # For simplicity, let's assume direct date match or closest previous day's close price.
        # Adjust dividend dates to be date-only for easier matching.
        dividends_df['Date'] = dividends_df['Date'].dt.tz_convert(None).dt.floor('D')
        hist_prices['Date'] = hist_prices['Date'].dt.floor('D')

        merged_df = pd.merge_asof(
            dividends_df.sort_values('Date'),
            hist_prices[['Date', 'Close']].sort_values('Date'),
            on='Date',
            direction='nearest' # Use nearest available price if exact date not found
        )

        # Calculate dividend as a percentage of close price
        # Handle cases where 'Close' might be NaN if no price data is available for a date
        merged_df['Dividend_Percent_of_Price'] = (merged_df['Dividend'] / merged_df['Close'] * 100).round(2)
        merged_df = merged_df.rename(columns={'Close': 'Stock_Price_at_Dividend_Date'})

        return merged_df

    except Exception as e:
        return f"An error occurred: {e}"

# Example Usage:
ticker = "wfc"
dividend_data = get_historical_dividends(ticker)
print(f"\nHistorical dividend data for {ticker}:")
display(dividend_data.tail(40))


Historical dividend data for wfc:


,Date,Dividend,Stock_Price_at_Dividend_Date,Dividend_Percent_of_Price
177,2016-05-04,0.38,37.289448,1.02
178,2016-08-03,0.38,36.536667,1.04
179,2016-11-02,0.38,35.036583,1.08
180,2017-02-01,0.38,43.594158,0.87
181,2017-05-03,0.38,43.067722,0.88
182,2017-08-02,0.39,42.407772,0.92
183,2017-11-02,0.39,44.981823,0.87
184,2018-02-01,0.39,52.484669,0.74
185,2018-05-03,0.39,41.603226,0.94
186,2018-08-09,0.43,47.518120,0.90


In [ ]:
import pandas as pd
import yfinance as yf

def get_historical_financial_data(ticker_symbol):
    """
    Fetches historical Total Assets, Net Income, and a proxy for Overhead
    (Operating Expenses) for a given ticker symbol.
    """
    try:
        ticker = yf.Ticker(ticker_symbol)

        # Fetch balance sheet data for Total Assets
        balance_sheet = ticker.balance_sheet
        if balance_sheet.empty:
            assets_df = pd.DataFrame()
        else:
            # Total Assets is usually the last row, or explicitly available.
            # Let's try to get 'Total Assets' directly if available, else infer.
            # yfinance financial statement keys can vary, so we'll check common ones.
            assets_row_name = None
            for col in balance_sheet.index:
                if 'Total Assets' in col:
                    assets_row_name = col
                    break

            if assets_row_name:
                assets_df = balance_sheet.loc[[assets_row_name]].T
                assets_df.index = assets_df.index.normalize()
                assets_df.columns = ['Total Assets']
            else:
                print(f"Warning: 'Total Assets' not found in balance sheet for {ticker_symbol}. Available rows: {balance_sheet.index.tolist()}")
                assets_df = pd.DataFrame()

        # Fetch income statement data for Net Income and Operating Expenses (as overhead proxy)
        income_statement = ticker.financials
        if income_statement.empty:
            income_overhead_df = pd.DataFrame()
        else:
            # Common names for Net Income and Operating Expenses
            net_income_row_name = None
            for col in income_statement.index:
                if 'Net Income' in col:
                    net_income_row_name = col
                    break

            operating_expenses_row_name = None
            for col in income_statement.index:
                if 'Operating Expenses' in col:
                    operating_expenses_row_name = col
                    break

            data_to_extract = []
            if net_income_row_name: data_to_extract.append(net_income_row_name)
            if operating_expenses_row_name: data_to_extract.append(operating_expenses_row_name)

            if data_to_extract:
                income_overhead_df = income_statement.loc[data_to_extract].T
                income_overhead_df.index = income_overhead_df.index.normalize()
                if net_income_row_name: income_overhead_df = income_overhead_df.rename(columns={net_income_row_name: 'Net Income'})
                if operating_expenses_row_name: income_overhead_df = income_overhead_df.rename(columns={operating_expenses_row_name: 'Overhead (Operating Expenses)'})
            else:
                print(f"Warning: 'Net Income' or 'Operating Expenses' not found in income statement for {ticker_symbol}. Available rows: {income_statement.index.tolist()}")
                income_overhead_df = pd.DataFrame()

        # Combine data into a single DataFrame
        if not assets_df.empty and not income_overhead_df.empty:
            combined_df = pd.merge(assets_df, income_overhead_df, left_index=True, right_index=True, how='outer')
        elif not assets_df.empty:
            combined_df = assets_df
        elif not income_overhead_df.empty:
            combined_df = income_overhead_df
        else:
            return f"No financial data found for {ticker_symbol}."

        combined_df.index.name = 'Date'
        combined_df = combined_df.sort_index()
        return combined_df

    except Exception as e:
        return f"An error occurred: {e}"

# Example Usage:
ticker_symbol = "dis"
financial_data = get_historical_financial_data(ticker_symbol)
print(f"\nHistorical financial data for {ticker_symbol}:")
display(financial_data)


Historical financial data for dis:


,Total Assets,Net Income
Date,,
2021-09-30,NaN,NaN
2022-09-30,2.036310e+11,3.193000e+09
2023-09-30,2.055790e+11,2.354000e+09
2024-09-30,1.962190e+11,4.972000e+09
2025-09-30,1.975140e+11,1.240400e+10


In [ ]:
import yfinance as yf

ticker = yf.Ticker('dis')
print(ticker.calendar)




{'Dividend Date': datetime.date(2026, 7, 22), 'Ex-Dividend Date': datetime.date(2026, 6, 30), 'Earnings Date': [datetime.date(2026, 5, 6)], 'Earnings High': 1.56, 'Earnings Low': 1.39, 'Earnings Average': 1.49101, 'Revenue High': 25452000000, 'Revenue Low': 24331305000, 'Revenue Average': 24828340450}
